# Activation Addition (ActAdd)

**Paper**: [Steering Language Models With Activation Engineering](https://arxiv.org/abs/2308.10248)

**Authors**: Alexander Matt Turner, Lisa Thiergart, Gavin Leech, David Udell, Juan J. Vazquez, Ulisse Mini, Monte MacDiarmid

Activation Addition (ActAdd) is a state control method that steers model behavior by computing a positional steering vector from a single pair of short prompts and adding it to the residual stream at a single layer.

## Method Parameters

| parameter              | type                | description                                                                                   |
| ---------------------- | ------------------- | --------------------------------------------------------------------------------------------- |
| `positive_prompt`      | `str`               | Prompt representing the desired direction (e.g., `"Love"`)                                    |
| `negative_prompt`      | `str`               | Prompt representing the opposite direction (e.g., `"Hate"`)                                   |
| `steering_vector`      | `SteeringVector`    | Pre-computed steering vector (alternative to prompts); must be extracted at the layer-input boundary |
| `layer_id`             | `int`               | Layer to inject at. If `None`, defaults to ~20% depth                                         |
| `multiplier`           | `float`             | Scaling coefficient (called `c` in the paper). Typical values range from 1 to 15              |
| `alignment`            | `int`               | Absolute token position at which injection begins (called `a` in the paper); row `t` of the vector is added at position `alignment + t`. Default: 0 |
| `normalize_vector`     | `bool`              | If `True`, L2-normalize each position's direction vector before applying                      |
| `use_norm_preservation`| `bool`              | If `True`, wrap the transform in `NormPreservingTransform` to prevent distribution shift      |

## Setup

If running this from a Google Colab notebook, please uncomment the following cell to install the toolkit. The following block is not necessary if running this notebook from a virtual environment where the package has already been installed.

In [1]:
# !git clone https://github.com/generative-computing/steerability.git
# %cd steerability

In [2]:
import re
import textwrap

import torch
from tabulate import tabulate
from transformers import AutoModelForCausalLM, AutoTokenizer

from steerability.algorithms.core.steering_pipeline import SteeringPipeline
from steerability.algorithms.state_control.act_add.control import ActAdd
from steerability.algorithms.state_control.common.estimators import SinglePairEstimator

For this demonstration, we use `Qwen2.5-1.5B`. Since ActAdd works with raw continuation prompts, we use the base model rather than the instruction-tuned variant. We load the model and tokenizer once and share them across the baseline and both steering pipelines.

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = model.device

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Baseline versus steered behavior for the sentiment example will be studied using the following test prompts.

In [4]:
test_prompts = [
    "I hate you because",
    "I think you're",
    "My favorite thing about life is",
    "I went up to my friend and said",
]

## Baseline Model Behavior

We first generate responses from the (unsteered) baseline model. Note that we reset the random seed before each generation so that the sampled completions are reproducible.

In [5]:
gen_params = {
    "max_new_tokens": 30,
    "do_sample": True,
    "temperature": 1.0,
    "top_p": 0.3,
    "repetition_penalty": 1.1,
    "pad_token_id": tokenizer.eos_token_id,
}

baseline_responses = []
for prompt in test_prompts:
    torch.manual_seed(1)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**enc, **gen_params)
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    baseline_responses.append(response)

print("Baseline completions:\n")
for prompt, response in zip(test_prompts, baseline_responses):
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")

Baseline completions:

Prompt: I hate you because
Response: I hate you because I love you

The title of this post is a line from the song “You’re Gonna Miss Me” by Billy Joel. The lyrics are

Prompt: I think you're
Response: I think you're going to have a lot of trouble getting the right answer. You see, there are two different ways that we can look at this problem.\nOne

Prompt: My favorite thing about life is
Response: My favorite thing about life is the people I get to meet. The most memorable ones are those who share my love of food, wine and travel.
I have a very close friend

Prompt: I went up to my friend and said
Response: I went up to my friend and said "Hey, I have a question for you." What did he say?
A. No.
B. Yes.
C. Sorry.
D. OK



## Token alignment of the contrast pairs

ActAdd takes the per-position difference between the residual streams of the two prompts, which requires the prompts to be aligned token by token. Following Algorithm 1 of the paper, the estimator right-pads the shorter prompt with attended space tokens. The rows of the vector beyond the shorter prompt are therefore differences between a trailing space and the remaining tokens of the longer prompt. Under Qwen's tokenizer `Hate` is two tokens and `Love` is one, which gives the sentiment vector two rows (the second against a space). The wedding pair spans five and seven tokens, which places rows 5 and 6 of its vector against spaces.

In [6]:
contrast_pairs = [
    ("Love", "Hate"),
    ("I talk about weddings constantly", "I do not talk about weddings constantly"),
]
for positive, negative in contrast_pairs:
    positive_ids = tokenizer.encode(positive, add_special_tokens=False)
    negative_ids = tokenizer.encode(negative, add_special_tokens=False)
    print(f"{positive!r}: {len(positive_ids)} tokens; {negative!r}: {len(negative_ids)} tokens")

'Love': 1 tokens; 'Hate': 2 tokens
'I talk about weddings constantly': 5 tokens; 'I do not talk about weddings constantly': 7 tokens


## Sentiment Steering

As in the original paper, we demonstrate sentiment steering using a "Love" vs "Hate" prompt pair, applied at layer 2 with a multiplier of 8. The `alignment` argument gives the absolute token position where injection begins; row `t` of the steering vector is added at position `alignment + t`. Qwen's tokenizer prepends no special tokens to the prompt, so `alignment=0` places the steering vector on the first prompt tokens. The vector has two rows (see the token counts above) and therefore covers the first two prompt tokens.

In [7]:
act_add_sentiment = ActAdd(
    positive_prompt="Love",
    negative_prompt="Hate",
    layer_id=2,
    multiplier=8,
    alignment=0,
)

sentiment_pipeline = SteeringPipeline(
    model=model,
    tokenizer=tokenizer,
    controls=[act_add_sentiment],
)
sentiment_pipeline.steer()

In [8]:
sentiment_responses = []
for prompt in test_prompts:
    torch.manual_seed(1)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = sentiment_pipeline.generate(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        return_full_sequence=True,
        **gen_params,
    )
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    sentiment_responses.append(response)

def wrap(text, width=50):
    return '\n'.join(textwrap.wrap(text, width=width))

table_data = []
for i, prompt in enumerate(test_prompts):
    table_data.append([
        wrap(prompt, 20),
        wrap(baseline_responses[i], 40),
        wrap(sentiment_responses[i], 40),
    ])

print(tabulate(
    table_data,
    headers=["prompt", "baseline", "steered (Love - Hate)"],
    tablefmt="grid",
))

+--------------------+------------------------------------------+-----------------------------------------+
| prompt             | baseline                                 | steered (Love - Hate)                   |
+====================+==========================================+=========================================+
| I hate you because | I hate you because I love you  The title | I hate you because you are a good       |
|                    | of this post is a line from the song     | person. You are the best thing in my    |
|                    | “You’re Gonna Miss Me” by Billy Joel.    | life. You have been there for me when I |
|                    | The lyrics are                           | needed someone to talk to, and          |
+--------------------+------------------------------------------+-----------------------------------------+
| I think you're     | I think you're going to have a lot of    | I think you're going to love this one.  |
|                    | troub

## Topic Steering

ActAdd can also be used to steer the model toward specific topics. We use the wedding example from the paper, intervening at layer 3 with `alignment=0` as before. The pair spans five and seven tokens, which gives the vector seven rows (rows 5 and 6 are differences against trailing spaces). The injection window therefore covers the first seven tokens of the sequence. The multiplier is chosen by the sweep in the next section. We evaluate on prompts of at least seven tokens, so that the whole window lies within the prompt, and generate baseline completions for these prompts first.

In [9]:
topic_prompts = [
    "I went up to my friend and said",
    "Yesterday my sister called to tell me about",
    "The best part of my weekend was when",
    "Last night at dinner my parents told us",
]

topic_baseline_responses = []
for prompt in topic_prompts:
    torch.manual_seed(1)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**enc, **gen_params)
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    topic_baseline_responses.append(response)

## Choosing the multiplier

We sweep the multiplier at layer 3 and sample eight completions per prompt (seeds 0 to 7) for each value. We then count the completions that mention a wedding-related word. The unsteered model gives the reference rate.

In [10]:
TOPIC_LAYER = 3
MULTIPLIERS = [1, 2, 4, 6, 8]
N_SEEDS = 8
WEDDING_WORDS = re.compile(r"\b(wedding|weddings|bride|groom|marr(?:y|ied|iage)|honeymoon|vows?)\b", re.IGNORECASE)

def wedding_rate(pipeline):
    mentions = 0
    for prompt in topic_prompts:
        enc = tokenizer(prompt, return_tensors="pt").to(device)
        for seed in range(N_SEEDS):
            torch.manual_seed(seed)
            output_ids = pipeline.generate(input_ids=enc.input_ids, attention_mask=enc.attention_mask, **gen_params)
            text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            mentions += bool(WEDDING_WORDS.search(text))
    return mentions / (len(topic_prompts) * N_SEEDS)

unsteered = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[])
unsteered.steer()
sweep_rows = [["unsteered", wedding_rate(unsteered)]]
for multiplier in MULTIPLIERS:
    control = ActAdd(
        positive_prompt="I talk about weddings constantly",
        negative_prompt="I do not talk about weddings constantly",
        layer_id=TOPIC_LAYER,
        multiplier=multiplier,
        alignment=0,
    )
    pipeline = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[control])
    pipeline.steer()
    sweep_rows.append([f"multiplier = {multiplier}", wedding_rate(pipeline)])

print(tabulate(sweep_rows, headers=["configuration", "wedding mentions"], floatfmt=".2f", tablefmt="grid"))

+-----------------+--------------------+
| configuration   |   wedding mentions |
+=================+====================+
| unsteered       |               0.03 |
+-----------------+--------------------+
| multiplier = 1  |               0.22 |
+-----------------+--------------------+
| multiplier = 2  |               0.72 |
+-----------------+--------------------+
| multiplier = 4  |               0.28 |
+-----------------+--------------------+
| multiplier = 6  |               0.22 |
+-----------------+--------------------+
| multiplier = 8  |               0.41 |
+-----------------+--------------------+


The wedding rate peaks at multiplier 2 (0.72, against 0.03 for the unsteered model). At larger multipliers the completions more often turn into multiple-choice exercises or lose coherence, and fewer of them mention a wedding. We therefore use multiplier 2 for the topic example.

In [11]:
TOPIC_MULTIPLIER = 2

In [12]:
act_add_topic = ActAdd(
    positive_prompt="I talk about weddings constantly",
    negative_prompt="I do not talk about weddings constantly",
    layer_id=TOPIC_LAYER,
    multiplier=TOPIC_MULTIPLIER,
    alignment=0,
)

topic_pipeline = SteeringPipeline(
    model=model,
    tokenizer=tokenizer,
    controls=[act_add_topic],
)
topic_pipeline.steer()

In [13]:
topic_responses = []
for prompt in topic_prompts:
    torch.manual_seed(1)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = topic_pipeline.generate(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        return_full_sequence=True,
        **gen_params,
    )
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    topic_responses.append(response)

table_data = []
for i, prompt in enumerate(topic_prompts):
    table_data.append([
        wrap(prompt, 20),
        wrap(topic_baseline_responses[i], 40),
        wrap(topic_responses[i], 40),
    ])

print(tabulate(
    table_data,
    headers=["prompt", "baseline", "steered (weddings)"],
    tablefmt="grid",
))

+----------------------+------------------------------------------+------------------------------------------+
| prompt               | baseline                                 | steered (weddings)                       |
+======================+==========================================+==========================================+
| I went up to my      | I went up to my friend and said "Hey, I  | I went up to my friend and said I had a  |
| friend and said      | have a question for you." What did he    | wedding in 1970. It was a big deal for   |
|                      | say? A. No. B. Yes. C. Sorry. D. OK      | me, but it wasn't really anything        |
|                      |                                          | special. I got married                   |
+----------------------+------------------------------------------+------------------------------------------+
| Yesterday my sister  | Yesterday my sister called to tell me    | Yesterday my sister called to tell me    |
|

## Dropping the padded rows

The estimator follows Algorithm 1 of the paper, which right-pads the shorter prompt of a pair. Rows 5 and 6 of the wedding vector are therefore differences between the remaining tokens of the longer prompt and trailing spaces rather than differences between the two prompts. To see what these rows contribute, we fit the vector with `SinglePairEstimator`, keep its first five rows (the length of the shorter prompt), and pass the result to `ActAdd` through `steering_vector`. The table below lists the norm of each row at the topic layer.

In [14]:
full_vector = SinglePairEstimator().fit(
    model,
    tokenizer,
    positive_prompt="I talk about weddings constantly",
    negative_prompt="I do not talk about weddings constantly",
    layer_ids=[TOPIC_LAYER],
)
shorter_length = min(
    len(tokenizer.encode("I talk about weddings constantly", add_special_tokens=False)),
    len(tokenizer.encode("I do not talk about weddings constantly", add_special_tokens=False)),
)
trimmed_vector = full_vector.clone()
trimmed_vector.directions = {
    layer_id: direction[:shorter_length] for layer_id, direction in trimmed_vector.directions.items()
}

norm_rows = [[row, f"{norm:.1f}"] for row, norm in enumerate(full_vector.directions[TOPIC_LAYER].norm(dim=-1).tolist())]
print(tabulate(norm_rows, headers=["row", f"norm at layer {TOPIC_LAYER}"], tablefmt="grid"))
print(f"\nfull vector: {full_vector.num_tokens} rows; trimmed vector: {trimmed_vector.num_tokens} rows")

+-------+-------------------+
|   row |   norm at layer 3 |
+=======+===================+
|     0 |               0   |
+-------+-------------------+
|     1 |              35.8 |
+-------+-------------------+
|     2 |              34.1 |
+-------+-------------------+
|     3 |              47.3 |
+-------+-------------------+
|     4 |              37.9 |
+-------+-------------------+
|     5 |              44.5 |
+-------+-------------------+
|     6 |              39.5 |
+-------+-------------------+

full vector: 7 rows; trimmed vector: 5 rows


We repeat the multiplier sweep with the trimmed vector, using the same prompts, seeds, and generation settings, and place its rates beside those of the full vector.

In [15]:
comparison_rows = [["unsteered", sweep_rows[0][1], sweep_rows[0][1]]]
for multiplier, (_, full_rate) in zip(MULTIPLIERS, sweep_rows[1:]):
    control = ActAdd(steering_vector=trimmed_vector, layer_id=TOPIC_LAYER, multiplier=multiplier, alignment=0)
    pipeline = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[control])
    pipeline.steer()
    comparison_rows.append([f"multiplier = {multiplier}", full_rate, wedding_rate(pipeline)])

print(tabulate(
    comparison_rows,
    headers=["configuration", "full vector (7 rows)", "trimmed vector (5 rows)"],
    floatfmt=".2f",
    tablefmt="grid",
))

+-----------------+------------------------+---------------------------+
| configuration   |   full vector (7 rows) |   trimmed vector (5 rows) |
+=================+========================+===========================+
| unsteered       |                   0.03 |                      0.03 |
+-----------------+------------------------+---------------------------+
| multiplier = 1  |                   0.22 |                      0.31 |
+-----------------+------------------------+---------------------------+
| multiplier = 2  |                   0.72 |                      0.75 |
+-----------------+------------------------+---------------------------+
| multiplier = 4  |                   0.28 |                      0.16 |
+-----------------+------------------------+---------------------------+
| multiplier = 6  |                   0.22 |                      0.25 |
+-----------------+------------------------+---------------------------+
| multiplier = 8  |                   0.41 |       

The two vectors give similar wedding rates at multipliers 1, 2, and 6. At multiplier 2 the trimmed vector reaches 0.75 (0.72 for the full vector), while at multipliers 4 and 8 the full vector is higher. Note that the row-0 norm of both vectors is zero, since the two prompts begin with the same token. We compare the two vectors at multiplier 2 on the topic prompts below.

In [16]:
trimmed_control = ActAdd(
    steering_vector=trimmed_vector,
    layer_id=TOPIC_LAYER,
    multiplier=TOPIC_MULTIPLIER,
    alignment=0,
)
trimmed_pipeline = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[trimmed_control])
trimmed_pipeline.steer()

trimmed_responses = []
for prompt in topic_prompts:
    torch.manual_seed(1)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = trimmed_pipeline.generate(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        return_full_sequence=True,
        **gen_params,
    )
    trimmed_responses.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))

table_data = [
    [wrap(prompt, 20), wrap(topic_responses[i], 40), wrap(trimmed_responses[i], 40)]
    for i, prompt in enumerate(topic_prompts)
]
print(tabulate(
    table_data,
    headers=["prompt", "full vector (7 rows)", "trimmed vector (5 rows)"],
    tablefmt="grid",
))

+----------------------+------------------------------------------+------------------------------------------+
| prompt               | full vector (7 rows)                     | trimmed vector (5 rows)                  |
+======================+==========================================+==========================================+
| I went up to my      | I went up to my friend and said I had a  | I went up to my friend and said I was    |
| friend and said      | wedding in 1970. It was a big deal for   | going to get married. He laughed at me,  |
|                      | me, but it wasn't really anything        | but he didn't say anything bad about it. |
|                      | special. I got married                   | He just said that I should be happy      |
+----------------------+------------------------------------------+------------------------------------------+
| Yesterday my sister  | Yesterday my sister called to tell me    | Yesterday my sister called to tell me    |
|

Two of the four completions from the full vector contain doubled spaces after a sentence, while none of the trimmed-vector completions do. The trimmed vector mentions a wedding in three of the four completions, against four of four for the full vector.

## Summary

This notebook demonstrated Activation Addition (ActAdd) for lightweight behavior steering:

1. ActAdd computes a positional steering vector from two short prompts.
2. The sentiment example showed how a "Love" vs "Hate" contrast at layer 2 shifts the tone of the completions toward positive.
3. The topic example raised the share of completions that mention weddings from 0.03 (unsteered) to 0.72 at layer 3 and multiplier 2. Dropping the two padded rows of the vector gives a similar rate (0.75) and removes the doubled spaces from the example completions.

ActAdd trades off statistical robustness (using more than a single prompt pair) for speed and simplicity, compared to contrastive activation addition (CAA) which aggregates over many pairs. The positional nature of the steering vector (row `t` is added at absolute position `alignment + t`, rather than broadcasting one vector over all positions) allows fine-grained control over where in the sequence the steering takes effect. The estimator right-pads the shorter prompt of a pair with attended space tokens. A pair of unequal token length therefore produces rows that compare a trailing space with the tokens of the longer prompt, and pairs of equal length have no such rows.